In [ ]:
from dotenv import load_dotenv
import os
import requests

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

from langgraph.prebuilt import create_react_agent

load_dotenv()

# --------------------------------------------------
# LLM
# --------------------------------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

# --------------------------------------------------
# Tools
# --------------------------------------------------

search_tool = DuckDuckGoSearchRun()

WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")


@tool
def get_weather_data(city: str) -> dict:
    """
    Get current weather for a city using WeatherStack.
    """

    if not WEATHERSTACK_API_KEY:
        return {
            "success": False,
            "error": "WEATHERSTACK_API_KEY is missing."
        }

    try:
        response = requests.get(
            "https://api.weatherstack.com/current",
            params={
                "access_key": WEATHERSTACK_API_KEY,
                "query": city,
            },
            timeout=20,
        )

        response.raise_for_status()

        data = response.json()

        if "error" in data:
            return {
                "success": False,
                "error": data["error"].get("info", "Unknown WeatherStack error"),
            }

        return {
            "success": True,
            "weather": data,
        }

    except Exception as e:
        return {
            "success": False,
            "error": str(e),
        }


# --------------------------------------------------
# Create Agent
# --------------------------------------------------

agent = create_react_agent(
    model=llm,
    tools=[
        search_tool,
        get_weather_data,
    ],
)


# --------------------------------------------------
# Helper
# --------------------------------------------------

def extract_text(content):
    """
    Convert Gemini content blocks into a plain string.
    """

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        texts = []

        for block in content:
            if isinstance(block, dict):

                if block.get("type") == "text":
                    texts.append(block.get("text", ""))

                elif "text" in block:
                    texts.append(block["text"])

        return "\n".join(texts)

    return str(content)


# --------------------------------------------------
# Chat Loop
# --------------------------------------------------

while True:

    question = input("\nQuestion: ")

    if question.lower() == "exit":
        break

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        }
    )

    final_message = response["messages"][-1]

    print("\nAnswer:\n")
    print(extract_text(final_message.content))

C:\Users\rudra\AppData\Local\Temp\ipykernel_10932\1276632584.py:52: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(



Answer:

[{'type': 'text', 'text': 'I am sorry, I cannot get the current weather for Surat. The weather API is not working correctly.', 'extras': {'signature': 'CukBARFNMg9cj2v+wW/TV+8qvCUsK23bv3sSHH4FTRHP0rewXe79k8TS/4//NfGLelHytVPWPCsLQx7QQE9Ib1xeCoWf0+4aZL8twQPLlnLy2ZXuzMvpP9A6Ch1lurHTl1cm7G8AqEKh6VD4F35ViwIXeQnegyeniM/bhqwtdsy25U/j8h/k9vklV31HabuQBBVyHlhTHAn3d1UJ8yhH9BdQiQrHQ+iwlCRjBASyILYJqdeqlU12FPsL0BmbUIH1SbtD/7SMq39F4aU9swOObZob+B3zIuXHy9KRM+L5v1NOw5Rl2765aHhGeqw='}}]
